In [1]:
from langgraph.graph import StateGraph, START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

e:\Lang_Chain\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CrashState(TypedDict):
    input:str
    step1:str
    step2:str

define steps

In [3]:
def step_1(state:CrashState)->CrashState:
    print("Step 1 Executed")
    return {"step1":"done","input":state["input"]}

In [4]:
def step_2(state:CrashState)->CrashState:
    print("Step2 hanging .....now manually interrupt from the notebook toolbar (STOP Button)")
    time.sleep(1000)
    return {"step2":"Done"}

In [5]:
def step_3(state:CrashState)->CrashState:
    print("Step3 Executed")
    return {"done":True}

Build Graph

In [12]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")

builder.add_edge("step_1","step_2")
builder.add_edge("step_2","step_3")
builder.add_edge("step_3",END)


In [13]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("Running graph : please interrupt manually during step2......")
    graph.invoke({"input:":"Start"},config = {"configurable":{"thread_id":"thread-1"}})
except KeyboardInterrupt:
    print("Kernel manually interrupted(crash stimulated)")

# Re-run to show fault-tolerant resume
print("\n Re-running the graph to demonstrate fault tolernce.....")
final_state=graph.invoke(None, config={"configurable":{"thread_id":"thread-1"}})
list(graph.get_state_history({"configurable":{"thread_id":"thread-1"}}))